# 01 – Exploring the OpenMensa API

**Goal:** understand what data the API gives us before writing the real scripts.

## My notes from browsing the API
- A canteen has: id, name, city, address, coordinates
- A meal has: id, name, category, prices, notes
- Prices are split into: students, employees, pupils, others
- Heidelberg has current data
- Old dates: TODO (write what you found)

In [24]:
import requests
import pandas as pd

BASE_URL = "https://openmensa.org/api/v2"

In [25]:
response = requests.get(f"{BASE_URL}/canteens")

print("Status:", response.status_code)

canteens = response.json()
print("Type:", type(canteens))
print("How many:", len(canteens))
print("First canteen:", canteens[0])

Status: 200
Type: <class 'list'>
How many: 500
First canteen: {'id': 1, 'name': 'Mensa UniCampus Magdeburg', 'city': 'Magdeburg', 'address': 'Pfälzer Str. 1, 39106 Magdeburg', 'coordinates': [52.139618827301895, 11.647599935531616]}


## Is that really all canteens?
My guess: I got 500 canteens. That seems like a lot, but it's exactly 500, which is suspiciously round. Maybe the API only sends 500 per page and there are more

In [26]:
for key, value in response.headers.items():
    print(key, ":", value)

# "x-total-pages" = 3 pages. Total canteens = 1306, 500 per page.
# So my first request only got page 1 (500 of 1306 canteens).

Server : nginx
Date : Thu, 08 Oct 2026 00:34:02 GMT
Content-Type : application/json; charset=utf-8
Transfer-Encoding : chunked
Connection : keep-alive
Vary : Accept-Encoding, Origin
x-frame-options : SAMEORIGIN
x-xss-protection : 0
x-content-type-options : nosniff
x-permitted-cross-domain-policies : none
referrer-policy : strict-origin-when-cross-origin
x-om-api-version : 2
link : <https://openmensa.org/api/v2/canteens?page=1>; rel="first", <https://openmensa.org/api/v2/canteens?page=2>; rel="next", <https://openmensa.org/api/v2/canteens?page=3>; rel="last"
x-total-pages : 3
x-total-count : 1306
x-per-page : 500
x-current-page : 1
etag : W/"2322dfbbe758ee90d98a456ffb86d678"
cache-control : max-age=0, private, must-revalidate
feature-policy : camera 'none'; gyroscope 'none'; microphone 'none'; usb 'none'; fullscreen 'self'; payment 'none'
content-security-policy : default-src 'self'; connect-src 'self'; font-src 'self' https: data:; img-src 'self' https: data:; object-src 'none'; script

In [27]:
all_canteens = []
page = 1
total_pages = int(response.headers["x-total-pages"])

while page <= total_pages:
    r = requests.get(f"{BASE_URL}/canteens", params={"page": page})
    all_canteens.extend(r.json())
    print(f"Page {page}: got {len(r.json())} canteens")
    page = page + 1

print("Total canteens:", len(all_canteens))

Page 1: got 500 canteens
Page 2: got 500 canteens
Page 3: got 306 canteens
Total canteens: 1306


In [28]:
df_canteens = pd.DataFrame(all_canteens)
df_canteens.head()

,id,name,city,address,coordinates
0,1,Mensa UniCampus Magdeburg,Magdeburg,"Pfälzer Str. 1, 39106 Magdeburg","[52.139618827301895, 11.647599935531616]"
1,2,Magdeburg Herrenkurg,Magdeburg,"Breitscheidstr. 2, 39114 Magdeburg","[52.14006658530411, 11.67566657066345]"
2,3,Mensa Stendal,Stendal,"Osterburger Str. 25, 39567 Stendal","[52.615015758022146, 11.849302053451538]"
3,4,Mensa Wernigerode,Wernigerode,"Friedrichstraße 57 – 59, 38855 Wernigerode","[51.8257591770436, 10.760003328323364]"
4,5,DomCafete Halberstadt,Halberstadt,"Am Domplatz 16, 38820 Halberstadt","[51.89534754646133, 11.047461032867432]"


In [29]:
df_canteens["city"].value_counts().head(10)

city
Hamburg                          60
Luxembourg                       53
Wien (Österreich)                44
Nürnberg                         25
Esch-sur-Alzette (Luxembourg)    24
Berlin                           24
Celle                            15
Leipzig                          14
Würzburg                         14
München                          13
Name: count, dtype: int64

In [30]:
heidelberg = df_canteens[df_canteens["city"] == "Heidelberg"]
heidelberg[["id", "name"]]

,id,name
187,279,"Heidelberg, Mensa Im Neuenheimer Feld 304"
189,281,"Heidelberg, Triplex-Mensa am Uniplatz"
190,282,"Heidelberg, zeughaus-Mensa im Marstall"


In [31]:
canteen_id = 279

r = requests.get(f"{BASE_URL}/canteens/{canteen_id}/days")
days = r.json()
days[:5]

[{'date': '2026-10-08', 'closed': False},
 {'date': '2026-10-09', 'closed': False},
 {'date': '2026-10-10', 'closed': True},
 {'date': '2026-10-11', 'closed': True},
 {'date': '2026-10-12', 'closed': False}]

In [32]:
date = "2026-10-08"

r = requests.get(f"{BASE_URL}/canteens/{canteen_id}/days/{date}/meals")
meals = r.json()

print("Number of meals:", len(meals))
meals[0]

Number of meals: 6


{'id': 28779892,
 'name': 'Tagessuppe\n',
 'category': 'A+B',
 'prices': {'students': 0.6, 'employees': 1.0, 'pupils': None, 'others': 1.5},
 'notes': ['green spelt soup\n']}

In [33]:
date = "2025-10-08"

r = requests.get(f"{BASE_URL}/canteens/{canteen_id}/days/{date}/meals")
meals_2025 = r.json()

print("Number of meals:", len(meals_2025))
meals[0]

Number of meals: 7


{'id': 28779892,
 'name': 'Tagessuppe\n',
 'category': 'A+B',
 'prices': {'students': 0.6, 'employees': 1.0, 'pupils': None, 'others': 1.5},
 'notes': ['green spelt soup\n']}

In [34]:
missing = 0

for meal in meals:
    if meal["prices"]["students"] is None:
        missing = missing + 1

print(f"{missing} of {len(meals)} meals have no student price")

0 of 6 meals have no student price


In [35]:
df_meals = pd.json_normalize(meals)
df_meals.head()

,id,name,category,notes,prices.students,prices.employees,prices.pupils,prices.others
0,28779892,Tagessuppe\n,A+B,[green spelt soup\n],0.60,1.00,None,1.5
1,28779893,Dessert Becher\n,A+B,[fresh fruit\n],0.60,1.00,None,1.5
2,28779894,Frischobst\n,A+B,[fresh fruit\n],0.80,1.00,None,1.0
3,28779895,Träumereien\n,A+B,[sweet delights\n],0.90,1.40,None,2.0
4,28779896,Schlemmerbuffet (je 100g)\nReichhaltige Auswah...,A+B,[Gourmet buffet each 100g Rich variety of fres...,0.99,1.35,None,2.0


## What I learned

- **Total canteens:** 1306, spread over 3 pages (500 per page). My first request only returned page 1, so I had to loop through all pages.

- **Cities with the most canteens:** Hamburg (60), Luxembourg (53), Wien (44), Nürnberg (25), Berlin (24).
  Surprise: OpenMensa is not only Germany. It includes Luxembourg and Austria, so I need to filter to German cities later.

- **Missing student prices:** 0 of 6 meals for Heidelberg, Mensa Im Neuenheimer Feld (2026-10-08). But the pupils price is always missing, probably because this Mensa doesn't serve school pupils.

- **Old dates available?** Yes. Asking for a specific past date (2025-10-08) returned 7 meals, even though the /days list only shows upcoming days. The soup cost €0.60 for students both then and now.

- **Surprises / problems:**
  - Names and notes end with "\n", which needs cleaning
  - The buffet is priced per 100g, not per meal, so it can't be compared directly with other meals
  - Category "A+B" doesn't tell me what kind of food it is, so for vegan/vegetarian I'll need to use the name and notes
  - Only 6 items on this day (soup, dessert, fruit, buffet), so I should check other canteens for main dishes

## Next steps
- Check if historical data exists
- Filter canteens to Germany